# Data pipeline: MassiveNuS κ → noisy, masked shear (tasks 1.2–1.6)

This notebook is a thin driver for `cosmicweb.data`. The logic and its tests live in `src/cosmicweb/data/` and `tests/`.

1. **Build once:** read MassiveNuS FITS maps (straight from the `.tar` on Drive), split them by line of sight into train/val/test, run the leakage checks, and save the clean κ maps.
2. **Generate on the fly:** `ShearMapDataset` turns each κ map into `[γ₁, γ₂, mask]` for any noise × mask level. Val/test inputs are fixed by seed; training can draw fresh noise each epoch.

See `docs/data.md` for the conventions and output layout.

## 1. Setup

A **CPU runtime is enough** for building the dataset; switch to a GPU runtime only for training.

In [ ]:
# Same steps as docs/TEAM_GUIDE.md section 7 ("Training on Colab or Kaggle").
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BRANCH = "feature/1.2-data-pipeline"  # change to "main" once the pull request is merged

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd /content
    if not Path("cosmic-web-unet").exists():
        !git clone -q https://github.com/astronova-cosmic-web/cosmic-web-unet.git
    %cd /content/cosmic-web-unet
    !git fetch -q origin && git checkout -q {BRANCH} && git pull -q --ff-only
    !pip install -q -e . astropy pyyaml
    # Do NOT reinstall torch or numpy on Colab: use the preinstalled versions.
    REPO = Path("/content/cosmic-web-unet")
else:
    REPO = Path.cwd().resolve().parent  # notebooks/ -> repo root

sys.path.insert(0, str(REPO / "src"))  # import without restarting the runtime after pip install -e
from cosmicweb.data import ShearMapDataset, build_dataset, load_config
from cosmicweb.data.qa import plot_examples

!git -C {REPO} log -1 --format="code version: %h %s"

## 2. Configuration

`DATASET_ROOT` is the Drive folder that holds the MassiveNuS `.tar`; it overrides the path in the config file. To change other settings, copy `configs/data.yaml` to Drive, edit it and point `CONFIG` at the copy.

In [ ]:
CONFIG = REPO / "configs" / "data.yaml"
DATASET_ROOT = ("/content/drive/MyDrive/Astronova/Research/DataSet" if IN_COLAB
                else os.environ.get("COSMICWEB_DATASET_ROOT"))

cfg = load_config(CONFIG, dataset_root=DATASET_ROOT)
DRIVE_OUT = cfg.build.output_dir  # permanent copy on Drive
# Work on the VM's local disk: Drive is slow for the random reads/writes of memory-mapped
# arrays. About 1 MiB per map, so ~10 GB for all 10,000 maps (Colab has ~70 GB free).
LOCAL_OUT = Path("/content/cosmicweb_data") / DRIVE_OUT.name if IN_COLAB else DRIVE_OUT

print("source     :", cfg.build.source, "(found)" if cfg.build.source.exists() else "(NOT FOUND: check DATASET_ROOT)")
print("drive copy :", DRIVE_OUT)
print("local copy :", LOCAL_OUT)
print("max_maps   :", cfg.build.max_maps, "(None = all maps)")

## 3. Build the dataset (once), or fetch an existing build

- **First time:** reads the `.tar` straight from Drive (no extraction needed), builds on local disk, then copies the result to Drive. Expect roughly 10–30 minutes for all maps. For a trial, set `build.max_maps: 50` and a different `output_dir` in your config copy.
- **Later sessions:** copies the finished build from Drive to local disk (a few minutes) instead of rebuilding.

An existing build is never overwritten unless you pass `overwrite=True`.

In [ ]:
import dataclasses
import json
import shutil

if (LOCAL_OUT / "dataset.json").exists():
    print("Using the local copy from this session.")
elif (DRIVE_OUT / "dataset.json").exists():
    print("Copying the existing build from Drive to local disk...")
    shutil.copytree(DRIVE_OUT, LOCAL_OUT, dirs_exist_ok=True)
else:
    local_cfg = dataclasses.replace(cfg, build=dataclasses.replace(cfg.build, output_dir=LOCAL_OUT))
    build_dataset(local_cfg)
    if LOCAL_OUT != DRIVE_OUT:
        print("Copying the build to Drive...")
        shutil.copytree(LOCAL_OUT, DRIVE_OUT, dirs_exist_ok=True)

meta = json.loads((LOCAL_OUT / "dataset.json").read_text())
print("maps per split :", meta["counts"])
print("map shape      :", meta["map_shape"], f"| pixel area {meta['pixel_area_arcmin2']:.4f} arcmin²")
print("leakage check  :", meta["leakage_check"])
print("skipped maps   :", len(meta["failed_maps"]), meta["failed_maps"] or "")
print("fingerprint    :", meta["fingerprint"][:16], "(quote this to identify the dataset)")

## 4. Visual QA across noise levels

In [ ]:
obs = cfg.observation
for noise in obs.noise_levels:
    print(f"{noise:>5}: sigma_pix = {obs.sigma_pix(noise, meta['pixel_area_arcmin2']):.4f}")

for noise in ("low", "mid", "high"):
    ds = ShearMapDataset(LOCAL_OUT, "val", obs, noise, "mid")
    plot_examples(ds, [0], DRIVE_OUT / f"qa_val_{noise}_mid.png")  # figures are saved on Drive

## 5. Loaders for training

In [ ]:
from torch.utils.data import DataLoader

NOISE, MASK = "high", "mid"
train_ds = ShearMapDataset(LOCAL_OUT, "train", obs, NOISE, MASK,
                           mode="random", patch_size=256, crops_per_map=4,
                           augment=True, resample_each_epoch=True)
val_ds = ShearMapDataset(LOCAL_OUT, "val", obs, NOISE, MASK, mode="tiles", patch_size=256)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)

for epoch in range(1):
    train_ds.set_epoch(epoch)  # new crops (and noise, since resample_each_epoch=True)
    batch = next(iter(train_loader))
    print(batch["x"].shape, batch["y"].shape)

## Notes

- **Inputs** are `[γ₁, γ₂, mask]` and **targets** are κ with the full-map mean removed. Both are multiplied by `observation.scale` (10); divide by it to get physical units.
- **Val/test** must use `augment=False` and `resample_each_epoch=False`, so their inputs never change between runs.
- **Noise levels** use the same random draws scaled by σ, so differences between levels come only from σ.
- **Local disk is wiped** when the Colab session ends; section 3 restores the build from Drive next time.